# 01b - Diagnostics
1. Is OU-DPS just too weak?
2. Where along the sampling is the OU guess `x_t / sqrt(alpha_bar_t)` useful?

Needs `ou_guidance/` and `resnet_cifar10.pt` in `My Drive/tfg_ou/`. Runtime: T4 GPU.
If the install cell asks to restart, restart and run from the top.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tfg_ou'      # code, checkpoint and results

In [ ]:
# pinned: a newer diffusers clashes with transformers
!pip install -q diffusers==0.35.2 transformers==4.55.2 "huggingface-hub>=0.34,<1.0" torch-fidelity

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/ou_guidance')
import helpers
helpers.ROOT = ROOT
from helpers import run, search, show_results, diagnose

## 1. Stronger rho (32 images)
Accuracy rises towards DPS: only tuning. Stays low: the guess is the limit.

In [ ]:
OUT = 'results_checks'
for rho in [2.0, 4.0, 8.0]:
    run('ou', n=32, out=OUT, rho=rho)
show_results(out=OUT, kinds=['unconditional', 'DPS', 'OU-DPS'])

## 1b. Mean-only
The cheap baseline: no backprop through the U-Net either (`mu` untuned).

In [ ]:
for mu in [0.25, 1.0, 4.0]:
    run('network', n=32, out=OUT, rho=0.0, mu=mu, n_iter=4, gamma=0.001)
show_results(out=OUT, kinds=['unconditional', 'DPS', 'mean-only', 'OU-DPS'])

## 2. The two guesses, step by step
Gradient size, clipped pixels (no gradient), agreement with the final class, distance between the guesses. Left: noise (t = 999); right: clean (t = 0).

In [ ]:
from IPython.display import Image
Image(diagnose(label=8, n=64))